In [1]:
pip install pandas scikit-learn werkzeug sqlalchemy flask_admin flask_migrate flask_rq2 flask_compress dnachisel openpyxl matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
from io import BytesIO
from datetime import datetime, UTC, timedelta
import traceback
import json
import io
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from werkzeug.exceptions import BadRequest
from pathlib import Path
import joblib
import matplotlib.pyplot
import statistics

#from app.helpers.fold_storage_manager import FoldStorageManager
from app.helpers.sequence_util import (
    get_measured_and_unmeasured_mutant_seq_ids,
    get_loci_set,
    process_and_validate_evolve_input_files,
)




def train_model(wt_aa_seq,raw_activity_df,raw_embedding_df,model):
    activity_df, embedding_df = process_and_validate_evolve_input_files(
                wt_aa_seq, raw_activity_df, raw_embedding_df
            )
    measured_mutants, unmeasured_mutants = (
                get_measured_and_unmeasured_mutant_seq_ids(activity_df, embedding_df)
            )
    X_train = np.vstack(
                [json.loads(x) for x in embedding_df.loc[activity_df.index].embedding]
            )
    y_train = activity_df.activity.to_numpy()
    #model = RandomForestRegressor(
    #    n_estimators=100,
    #    criterion="friedman_mse",
    #    max_depth=None,
    #    min_samples_split=2,
    #    min_samples_leaf=1,
    #    min_weight_fraction_leaf=0.0,
    #    max_features=1.0,
    #    max_leaf_nodes=None,
    #    min_impurity_decrease=0.0,
    #    bootstrap=True,
    #    oob_score=False,
    #    n_jobs=None,
    #    random_state=1,
    #    verbose=0,
    #    warm_start=False,
    #    ccp_alpha=0.0,
    #    max_samples=None,
    #)
    model.fit(X_train, y_train)
    try:
        all_mutants_embedding_array = np.vstack(
            [
                json.loads(x)
                for x in embedding_df.loc[
                    measured_mutants + unmeasured_mutants
                ].embedding
            ]
        )
        #print(all_mutants_embedding_array.shape)
        y_all_pred = model.predict(all_mutants_embedding_array)
        predicted_activity_df = pd.DataFrame(
            {
                "seq_id": measured_mutants + unmeasured_mutants,
                "predicted_activity": y_all_pred,
            }
        )
        predicted_activity_df.index = predicted_activity_df.seq_id
        predicted_activity_df["relevant_measured_mutants"] = (
            predicted_activity_df.seq_id.apply(
                lambda seq_id: " ".join(
                    [
                        m
                        for m in measured_mutants
                        if get_loci_set(m) & get_loci_set(seq_id)
                    ]
                )
            )
        )
        predicted_activity_df["actual_activity"] = predicted_activity_df.join(
            activity_df.groupby(level=0).activity.mean(), how="left"
        ).activity
        predicted_activity_df = predicted_activity_df.sort_values(
            "predicted_activity", ascending=False
        )
    except Exception as e:
        print(f"Failed to predict activities: {e}")
        raise
    predicted_activity_df.reset_index(drop=True,inplace=True)
    #predicted_activity_csv_path = evolve_directory / f"Round_{round_num}_predicted_activity.csv"
    #print(f"Storing predicted activities in {predicted_activity_csv_path}")
    #try:
    #    predicted_activity_df.to_csv(predicted_activity_csv_path, index=False)
    #except Exception as e:
    #    print(f"Failed to store predicted activities: {e}")
    #    raise
    return predicted_activity_df

def evaluate_predictions(predicted_activity_df,exp_activity_df,round_activity_df,num_var,round_num,evolve_directory):
    try:
        predict = predicted_activity_df["actual_activity"].isna()
        extract_predict = predicted_activity_df[predict]
        #print(extract_predict)
        top_var = extract_predict.iloc[0:num_var]
        #print(top_var)
        top_var_real = pd.merge(top_var,exp_activity_df,on='seq_id', how='inner')
        top_var_real = top_var_real[['seq_id','activity']]
        #top_var_csv_path = evolve_directory / f"Round_{round_num}_top_variants.xlsx"
        #top_var_real.to_excel(top_var_csv_path, index=False)
        next_round_activity = pd.concat([round_activity_df,top_var_real], ignore_index=True)
    except Exception as e:
        print(f"Failed to Evaluate Predictions")
        raise
    return next_round_activity

def evolve_simulation(wt_aa_seq,embeddings_path,exp_activity_file_path,num_var,model_type):
    exp_activity_df = pd.read_excel(exp_activity_file_path)
    raw_embedding_df = pd.read_csv(embeddings_path)
    raw_activity_df = exp_activity_df.sample(num_var)
    while not raw_activity_df['seq_id'].isin(raw_embedding_df['seq_id']).all():
        raw_activity_df = exp_activity_df.sample(num_var)
    #print(raw_activity_df)
    evolve_directory = Path("evolve") / Path(exp_activity_file_path).stem
    evolve_directory.mkdir(parents=True, exist_ok=True)
    init_var_path = evolve_directory / f"Round_0_variants.xlsx"
    raw_activity_df.to_excel(init_var_path, index=False)
    var_exp_data = [raw_activity_df.sort_values('activity',ascending=True)]
    top_percent_df = exp_activity_df.sort_values('activity',ascending=False)
    #print(top_percent_df)
    #print('this is the sorted activity')
    quantile = 0.90
    top_percent_activity = exp_activity_df['activity'].quantile(quantile)
    top_percent_df = exp_activity_df[exp_activity_df['activity'] >= top_percent_activity]
    #print('There are this many top variants')
    #print(top_percent_df.shape)
    #print(f'These are the top variants')
    #print(top_percent_df)
    #print(round(len(top_percent_df)*0.51))
    i = 1
    complete = True
    while complete:
        if i == 1:
            predicted_activity_df = train_model(wt_aa_seq,raw_activity_df,raw_embedding_df,model_type)
            current_round_activity_df = evaluate_predictions(predicted_activity_df,exp_activity_df,raw_activity_df,num_var,i,evolve_directory)
            #print(i)
            #print(current_round_activity_df)
        else:
            predicted_activity_df = train_model(wt_aa_seq,current_round_activity_df,raw_embedding_df,model_type)
            current_round_activity_df = evaluate_predictions(predicted_activity_df,exp_activity_df,current_round_activity_df,num_var,i,evolve_directory)
            #print(i)
            #print(top_percent_df['seq_id'].isin(current_round_activity_df['seq_id']).all())
        var_exp_data.append(current_round_activity_df.sort_values('activity',ascending=True))
        #if i == round_num:
        #    complete = False
        #    break
        num_top_var_df = pd.merge(top_percent_df,current_round_activity_df, on='seq_id')
        if len(num_top_var_df) >= (round(len(top_percent_df)*0.51)):
            print('test')
            complete = False
            break  
        print(f'Round {i} Evolution found {len(num_top_var_df)} top variants')
        i += 1
        #print(current_round_activity_df.shape)

        
    print(f'It took {i} rounds to get the majority of the top {quantile * 100} % of variants')
    #print(var_exp_data)
    return i
    
    

In [3]:
randomforest = RandomForestRegressor(
        n_estimators=100,
        criterion="friedman_mse",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        min_weight_fraction_leaf=0.0,
        max_features=1.0,
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
        bootstrap=True,
        oob_score=False,
        n_jobs=None,
        random_state=1,
        verbose=0,
        warm_start=False,
        ccp_alpha=0.0,
        max_samples=None,
    )


In [5]:
wt_aa_seq = 'MAKEDNIEMQGTVLETLPNTMFRVELENGHVVTAHISGKMRKNYIRILTGDKVTVELTPYDLSKGRIVFRSR'
activity_file_path = r'..\EvolveTest\kelsic_Round1.xlsx'
exp_activity_file_path = r'..\EvolveTest\Kelsic_Data_foldy.xlsx'
embeddings_path = r'..\EvolveTest\007426_embeddings_esmc_600m_wt_dms.csv'
num_var = 20
round_to_best = []
model_type = randomforest
for j in range(1,10):
    rounds = evolve_simulation(wt_aa_seq,embeddings_path,exp_activity_file_path,num_var,model_type)
    round_to_best.append(rounds)
print(statistics.median(round_to_best))
print(statistics.mean(round_to_best))

Round 1 Evolution found 3 top variants
Round 2 Evolution found 6 top variants
Round 3 Evolution found 13 top variants
Round 4 Evolution found 19 top variants
Round 5 Evolution found 27 top variants
Round 6 Evolution found 29 top variants
Round 7 Evolution found 34 top variants
Round 8 Evolution found 37 top variants
Round 9 Evolution found 42 top variants
Round 10 Evolution found 42 top variants
Round 11 Evolution found 44 top variants
Round 12 Evolution found 45 top variants
Round 13 Evolution found 50 top variants
Round 14 Evolution found 56 top variants
Round 15 Evolution found 59 top variants
Round 16 Evolution found 60 top variants
Round 17 Evolution found 63 top variants
Round 18 Evolution found 66 top variants
Round 19 Evolution found 71 top variants
Round 20 Evolution found 72 top variants
test
It took 21 rounds to get the majority of the top 90.0 % of variants
Round 1 Evolution found 2 top variants
Round 2 Evolution found 5 top variants
Round 3 Evolution found 7 top variants
R

KeyboardInterrupt: 